In [80]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')
from data_loader import load_raw_data

In [81]:
renewal_calls, billings, cc_calls, emails = load_raw_data()

1-Data cleaning of renewal_calls

In [82]:
renewal_calls.shape

(186534, 41)

In [83]:
renewal_calls.duplicated().sum()

np.int64(28812)

Removing duplicates from renewal_calls

In [84]:
renewal_calls = renewal_calls.drop_duplicates()
renewal_calls.shape

(157722, 41)

Dropping rows where Co_Ref is null because it acts as primary key to join with other datasets

In [85]:
renewal_calls['Co_Ref'].isnull().sum()

np.int64(4453)

In [86]:
renewal_calls = renewal_calls.dropna(subset=['Co_Ref'])
renewal_calls.shape

(153269, 41)

Standardizing Call_Direction column values in renewal_calls

In [87]:
renewal_calls['Call_Direction'].value_counts()

Call_Direction
OUT_BOUND    65852
Outbound     65616
IN_BOUND     11149
Inbound      10652
Name: count, dtype: int64

In [88]:
renewal_calls['Call_Direction'] = renewal_calls['Call_Direction'].replace({
    'OUT_BOUND': 'Outbound',
    'IN_BOUND': 'Inbound'
})
renewal_calls['Call_Direction'].value_counts()

Call_Direction
Outbound    131468
Inbound      21801
Name: count, dtype: int64

Converting proper datatype of dates in renewal_calls

In [89]:
renewal_calls['Call_Date'].dtype

<StringDtype(storage='python', na_value=nan)>

In [90]:
renewal_calls['Call_Date'].head(5)

0    29-01-2025
1    26-02-2025
2    24-01-2025
3    09-06-2025
4    20-08-2024
Name: Call_Date, dtype: str

In [91]:
renewal_calls['Call_Date'] = pd.to_datetime(renewal_calls['Call_Date'], format='%d-%m-%Y')
renewal_calls['Call_Date'].dtype

dtype('<M8[us]')

In [92]:
renewal_calls['Call_Date'].head(5)

0   2025-01-29
1   2025-02-26
2   2025-01-24
3   2025-06-09
4   2024-08-20
Name: Call_Date, dtype: datetime64[us]

Dropping columns with all nulls in renewal_calls

In [93]:
empty_columns = renewal_calls.columns[renewal_calls.isnull().all()]
renewal_calls = renewal_calls.drop(columns=empty_columns)

In [94]:
renewal_calls.shape

(153269, 40)

Dropping Analysed_Call in renewal_calls as it has only 1 unique value

In [95]:
renewal_calls['Analysed_Call'].nunique()

1

In [96]:
renewal_calls = renewal_calls.drop('Analysed_Call', axis=1)

Dropping columns with almost more than 90% null data

In [97]:
drop_columns = [
    'Argument_That_Convinced_Customer_to_Stay_Category',
    'Agent_Response_To_Cancel_Category',
    'Justification_Category',
    'Reason_For_Renewal_Category',
]
for col in drop_columns:
    print(f"Null percentage of {col}: ", (renewal_calls[col].isnull().sum()/len(renewal_calls))*100)

Null percentage of Argument_That_Convinced_Customer_to_Stay_Category:  98.86147883786023
Null percentage of Agent_Response_To_Cancel_Category:  96.86564145391436
Null percentage of Justification_Category:  91.7922084700755
Null percentage of Reason_For_Renewal_Category:  89.90337250194104


In [98]:
renewal_calls = renewal_calls.drop(columns=drop_columns)

In [99]:
renewal_calls.shape

(153269, 35)

Dropping irrelevant columns<br>
Call_Year-> can be derived from Call_Date

In [100]:
renewal_calls['Call_Year'].unique()

array([2025, 2024, 2026])

In [101]:
renewal_calls = renewal_calls.drop('Call_Year', axis=1)

In [102]:
renewal_calls.shape

(153269, 34)

2-Data cleaning of billings

In [103]:
billings.shape

(122082, 59)

In [104]:
billings.duplicated().sum()

np.int64(0)

Converting proper datatype of dates

In [105]:
billings['Renewal_Month'].head()

0    01-11-2024
1    01-08-2025
2    01-03-2025
3    01-06-2025
4    01-03-2025
Name: Renewal_Month, dtype: str

In [106]:
billings['Prospect_Renewal_Date'].head()

0    05-11-2024
1    09-08-2025
2    12-03-2025
3    29-06-2025
4    25-03-2025
Name: Prospect_Renewal_Date, dtype: str

In [107]:
billings['DateTime_Out'].head()

0    01-11-2024
1    01-08-2025
2    01-03-2025
3    01-06-2025
4    01-03-2025
Name: DateTime_Out, dtype: str

In [108]:
billings['Renewal_Month'] = pd.to_datetime(billings['Renewal_Month'], format='%d-%m-%Y')
billings['Prospect_Renewal_Date'] = pd.to_datetime(billings['Prospect_Renewal_Date'], format='%d-%m-%Y')
billings['DateTime_Out'] = pd.to_datetime(billings['DateTime_Out'], format='%d-%m-%Y')

In [109]:
billings['Renewal_Month'].head()

0   2024-11-01
1   2025-08-01
2   2025-03-01
3   2025-06-01
4   2025-03-01
Name: Renewal_Month, dtype: datetime64[us]

In [110]:
billings['Prospect_Renewal_Date'].head()

0   2024-11-05
1   2025-08-09
2   2025-03-12
3   2025-06-29
4   2025-03-25
Name: Prospect_Renewal_Date, dtype: datetime64[us]

Dropping column with 100% null values

In [111]:
billings['Last_Years_Date_Paid'].notna().sum()

np.int64(0)

In [112]:
billings = billings.drop('Last_Years_Date_Paid', axis=1)

In [113]:
billings.shape

(122082, 58)

Drop redundant columns<br>
1- Starting_Gross=Starting_Net+Starting_Vat<br>
2- DateTime_Out is same as Renewal_Month<br>
3- Anchor_Group is same as Connection_Group

In [114]:
check = billings['Starting_Net'] + billings['Starting_Vat']
print((check == billings['Starting_Gross']).all())

True


In [115]:
print((billings['DateTime_Out'] == billings['Renewal_Month']).all())

True


In [116]:
print((billings['Anchor_Group'] == billings['Connection_Group']).dropna().all())

False


In [117]:
diff = billings['Anchor_Group'] != billings['Connection_Group']
print("Anchor_Group vs Connection_Group:")
print(f"  Same: {(~diff).sum()}")
print(f"  Different: {diff.sum()}")
print(f"  Different %: {diff.mean()*100:.2f}%")

Anchor_Group vs Connection_Group:
  Same: 121956
  Different: 126
  Different %: 0.10%


In [118]:
diff = billings['Anchor_Group'] != billings['Connection_Group']


In [119]:
billings[diff2][['DateTime_Out', 'Renewal_Month']].head(10)
print(f"  Different: {diff.sum()}")

  Different: 126


In [120]:
billings[diff][['Anchor_Group', 'Connection_Group']].head(10)

,Anchor_Group,Connection_Group
46784,NaN,NaN
46785,NaN,NaN
46786,NaN,NaN
46787,NaN,NaN
46789,NaN,NaN
59007,NaN,NaN
59011,NaN,NaN
59014,NaN,NaN
59018,NaN,NaN
59019,NaN,NaN


In [121]:
billings = billings.drop(columns=['Starting_Vat', 'Starting_Gross', 'DateTime_Out', 'Anchor_Group'])
billings.shape

(122082, 54)

3-Data Cleaning of cc_calls

In [122]:
cc_calls.shape

(32882, 33)

Dropping duplicate columns

In [123]:
cc_calls.duplicated().sum()

np.int64(93)

In [124]:
cc_calls = cc_calls.drop_duplicates()
cc_calls.shape


(32789, 33)

Dropping rows with null Co_Ref

In [125]:
cc_calls['Co_Ref'].isnull().sum()

np.int64(1153)

In [126]:
cc_calls = cc_calls.dropna(subset=['Co_Ref'])
cc_calls.shape

(31636, 33)

Standardizing values in Direction column

In [127]:
cc_calls['Direction'].value_counts()


Direction
OUT_BOUND    24292
IN_BOUND      7344
Name: count, dtype: int64

In [128]:
cc_calls['Direction'] = cc_calls['Direction'].replace({
    'OUT_BOUND': 'Outbound',
    'IN_BOUND': 'Inbound'
})
cc_calls['Direction'].value_counts()


Direction
Outbound    24292
Inbound      7344
Name: count, dtype: int64

Converting proper datatype of dates

In [129]:
cc_calls['Call_Date'].head()

0    08-05-2025
1    25-11-2024
2    23-10-2024
3    13-01-2025
4    19-03-2025
Name: Call_Date, dtype: str

In [130]:
cc_calls['Call_Date'] = pd.to_datetime(cc_calls['Call_Date'], format='%d-%m-%Y')
cc_calls['Call_Date'].head()

0   2025-05-08
1   2024-11-25
2   2024-10-23
3   2025-01-13
4   2025-03-19
Name: Call_Date, dtype: datetime64[us]

Dropping irrelevant columns<br>
1- Analysed Calls-> have only one unique value<br>
2- Call_Year-> can be derived from Call_Year

In [131]:
cc_calls['Analysed_Call'].unique()

array([1])

In [132]:
cc_calls['Call_Year'].unique()

array([2025, 2024, 2026])

In [133]:
cc_calls = cc_calls.drop(columns=['Analysed_Call', 'Call_Year'])
cc_calls.shape

(31636, 31)

4-Data Cleaning of emails dataset

In [134]:
emails.shape

(123389, 27)

Dropping duplicate rows


In [135]:
emails.duplicated().sum()

np.int64(0)

In [136]:
emails['Co_Ref'].isnull().sum()

np.int64(0)

In [137]:
emails['Time_to_Renewal'].value_counts()

Time_to_Renewal
prior_year     40022
14_out         32493
45_out         28008
pre_renewal    22866
Name: count, dtype: int64

Dropping irrelevant columns<br>
year-> can be derived

In [138]:
emails['year'].unique()

array([2025, 2026, 2024])

In [139]:
emails = emails.drop('year', axis=1)
emails.shape

(123389, 26)

Saving cleaned datasets in data/cleaned

In [140]:
import os
os.makedirs('../data/cleaned', exist_ok=True)

renewal_calls.to_csv('../data/cleaned/renewal_calls_cleaned.csv', index=False)
billings.to_csv('../data/cleaned/billings_cleaned.csv', index=False)
cc_calls.to_csv('../data/cleaned/cc_calls_cleaned.csv', index=False)
emails.to_csv('../data/cleaned/emails_cleaned.csv', index=False)

print("All datasets saved to data/cleaned/")

All datasets saved to data/cleaned/
